<a href="https://colab.research.google.com/github/gpgovindapromod/aiml/blob/main/Logistic_Regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Upload both CSV files
uploaded = files.upload()

# Load datasets
weather_df = pd.read_csv("weather_prediction_dataset.csv")
labels_df = pd.read_csv("weather_prediction_bbq_labels.csv")

print("Weather dataset:", weather_df.shape)
print("Labels dataset:", labels_df.shape)

Saving weather_prediction_bbq_labels.csv to weather_prediction_bbq_labels (1).csv
Saving weather_prediction_dataset.csv to weather_prediction_dataset (1).csv
Weather dataset: (3654, 165)
Labels dataset: (3654, 18)


In [ ]:
# Merge weather data with BBQ weather labels
df = pd.merge(
    weather_df,
    labels_df[['DATE', 'BASEL_BBQ_weather']],
    on='DATE',
    how='inner'
)

# Convert target to 0 and 1
df['BASEL_BBQ_weather'] = df['BASEL_BBQ_weather'].astype(int)

# Features
features = [
    'BASEL_cloud_cover',
    'BASEL_humidity',
    'BASEL_pressure',
    'BASEL_global_radiation',
    'BASEL_precipitation',
    'BASEL_sunshine',
    'BASEL_temp_mean',
    'BASEL_temp_min',
    'BASEL_temp_max'
]

X = df[features]
y = df['BASEL_BBQ_weather']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Standardization
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Preprocessing completed.")
print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Preprocessing completed.
Training samples: 2923
Testing samples: 731


In [ ]:
# Create and train Logistic Regression
logistic_model = LogisticRegression(
    random_state=42,
    max_iter=1000
)

logistic_model.fit(X_train_scaled, y_train)

# Predictions
y_pred = logistic_model.predict(X_test_scaled)

# Metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)

print("=" * 50)
print("LOGISTIC REGRESSION CLASSIFICATION")
print("=" * 50)

print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1 Score  : {f1:.4f}")

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=['Not BBQ Weather', 'BBQ Weather'],
    zero_division=0
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

LOGISTIC REGRESSION CLASSIFICATION
Accuracy  : 0.9330
Precision : 0.8557
Recall    : 0.8877
F1 Score  : 0.8714

Classification Report:
                 precision    recall  f1-score   support

Not BBQ Weather       0.96      0.95      0.95       544
    BBQ Weather       0.86      0.89      0.87       187

       accuracy                           0.93       731
      macro avg       0.91      0.92      0.91       731
   weighted avg       0.93      0.93      0.93       731


Confusion Matrix:
[[516  28]
 [ 21 166]]


In [ ]:
# Install Gradio
!pip install -q gradio

import gradio as gr

# Prediction function
def predict_weather(
    cloud_cover,
    humidity,
    pressure,
    global_radiation,
    precipitation,
    sunshine,
    temp_mean,
    temp_min,
    temp_max
):
    # Create input DataFrame
    user_data = pd.DataFrame([[
        cloud_cover,
        humidity,
        pressure,
        global_radiation,
        precipitation,
        sunshine,
        temp_mean,
        temp_min,
        temp_max
    ]], columns=features)

    # Scale using the trained scaler
    user_scaled = scaler.transform(user_data)

    # Prediction
    prediction = logistic_model.predict(user_scaled)[0]
    probability = logistic_model.predict_proba(user_scaled)[0]

    if prediction == 1:
        result = "🌤️ BBQ WEATHER"
    else:
        result = "🌧️ NOT BBQ WEATHER"

    return (
        result,
        f"{probability[0] * 100:.2f}%",
        f"{probability[1] * 100:.2f}%"
    )


# Create UI
with gr.Blocks(title="Weather Prediction") as demo:

    gr.Markdown(
        """
        # 🌦️ Basel BBQ Weather Prediction
        ### Logistic Regression Classification
        Enter the weather conditions below and click **Predict**.
        """
    )

    with gr.Row():

        with gr.Column():
            cloud_cover = gr.Number(
                label="Cloud Cover (0-8)",
                value=4
            )

            humidity = gr.Number(
                label="Humidity",
                value=0.70
            )

            pressure = gr.Number(
                label="Pressure",
                value=1015
            )

            global_radiation = gr.Number(
                label="Global Radiation",
                value=150
            )

            precipitation = gr.Number(
                label="Precipitation",
                value=0
            )

        with gr.Column():
            sunshine = gr.Number(
                label="Sunshine",
                value=5
            )

            temp_mean = gr.Number(
                label="Mean Temperature (°C)",
                value=15
            )

            temp_min = gr.Number(
                label="Minimum Temperature (°C)",
                value=10
            )

            temp_max = gr.Number(
                label="Maximum Temperature (°C)",
                value=20
            )

    predict_button = gr.Button(
        "🔍 Predict Weather",
        variant="primary"
    )

    result = gr.Textbox(
        label="Prediction",
        interactive=False
    )

    not_bbq_probability = gr.Textbox(
        label="Not BBQ Weather Probability",
        interactive=False
    )

    bbq_probability = gr.Textbox(
        label="BBQ Weather Probability",
        interactive=False
    )

    predict_button.click(
        fn=predict_weather,
        inputs=[
            cloud_cover,
            humidity,
            pressure,
            global_radiation,
            precipitation,
            sunshine,
            temp_mean,
            temp_min,
            temp_max
        ],
        outputs=[
            result,
            not_bbq_probability,
            bbq_probability
        ]
    )

# Launch UI
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b3e6242662afa3d0e3.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
